In [ ]:
import importlib
import NeuralNetwork
import funcs
import plots

importlib.reload(NeuralNetwork)
importlib.reload(funcs)
importlib.reload(plots)
from NeuralNetwork import NeuralNetwork

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm
import copy
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device being used: {device.type}")

## Global parameters

In [ ]:
# model parameters
HIDDEN_LAYERS = [512, 256, 128, 64]
TRAIN_VAL_SPLIT = 0.8
NUM_WORKERS = 4
 
# initial training
N_TRAIN_EPOCHS = 15 if device.type == "cuda" else 8

# pruning loop
MAX_ALLOWED_ACC_DROP = 0.02
MAX_PRUNE_ROUNDS = 30
PRUNE_FRAC = 0.05
PRUNE_CON_FRAC = 0.1
REGROW_FRAC = 0.1
MIN_VAL_ACC = 0.9
N_RETRIAN_EPOCHS = 3

# clustering
N_CLUSTERS = 15

## Setting up data and initial model

In [ ]:
def rotate_image(x):
    return torch.rot90(x, k=-1, dims=[1, 2])

def flip_image(x):
    return torch.flip(x, dims=[2])

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(rotate_image),
    transforms.Lambda(flip_image),
])

# Download and load the training set
train_dataset = datasets.EMNIST(
    root="./data",       # where to store the data
    split="digits",      # "digits" for 0–9, other options: "letters", "balanced", etc.
    train=True,
    download=True,
    transform=transform
)

# Download and load the test set
test_dataset = datasets.EMNIST(
    root="./data",
    split="digits",
    train=False,
    download=True,
    transform=transform
)

In [ ]:
train_size = int(TRAIN_VAL_SPLIT * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])
print(f"train size: {train_size}, val size: {val_size}, test size: {len(test_dataset)}")

In [ ]:
# Create a DataLoader for batching
batch_size = 4000
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
# Create model and train
model = NeuralNetwork(hidden_sizes=HIDDEN_LAYERS, device=device)
model.train_model(train_loader=train_loader, val_loader=val_loader, epochs=N_TRAIN_EPOCHS)

## Prune Neurons and Retrain

In [ ]:
prune_parameters = (MAX_PRUNE_ROUNDS, PRUNE_FRAC, PRUNE_CON_FRAC, REGROW_FRAC, N_RETRIAN_EPOCHS, MAX_ALLOWED_ACC_DROP)
use_max_rounds = False if device.type == "cuda" else True

final_model = funcs.pruning(model, train_loader, val_loader, prune_parameters, use_max_rounds=use_max_rounds, mode='full')

In [ ]:
print(f"Test accuracy after pruning: {final_model.accuracy(val_loader):.2f}")

In [ ]:
torch.save(final_model, "pruned_model.pth")

## Activation analysis and Neuron Clustering

In [ ]:
if os.path.exists("pruned_model.pth"):
    final_model = torch.load("pruned_model.pth", weights_only=False)
    final_model.eval()
else:
    print("No final model saved on disk")

We create a new data loader that does not shuffle the data so our instances match up with our data

In [ ]:
analysis_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
analysis_model = copy.deepcopy(final_model)
analysis_model.eval()

In [ ]:
layer_data_analysis = analysis_model.get_layer_data(analysis_loader)

In [ ]:
for key, value in layer_data_analysis.items():
    print(f"{key} has shape: {value['post_activation'].shape}")

In [ ]:
hidden_layers = [k for k in layer_data_analysis.keys() if 'layer_' in k]

all_neuron_activations = []
for layer_name in hidden_layers:
    acts = layer_data_analysis[layer_name]['post_activation']
    all_neuron_activations.append(acts)

all_neuron_activations = torch.cat(all_neuron_activations, dim=1)

In [ ]:
layer_mapping = []
start_idx = 0
for layer_name in hidden_layers:
    n_neurons = layer_data_analysis[layer_name]['post_activation'].shape[1]
    end_idx = start_idx + n_neurons
    layer_mapping.append((layer_name, start_idx, end_idx))
    start_idx = end_idx

In [ ]:
labels = []
images = []
for X, y in tqdm(analysis_loader):
    labels.append(y)
    images.append(X)
labels = torch.cat(labels, dim=0)
images = torch.cat(images, dim=0)

## Clustering

In [ ]:
from sklearn.decomposition import NMF

NMF_SUBSAMPLE = 15000
activations_np = all_neuron_activations.numpy()  # [N_samples, N_neurons]

if activations_np.shape[0] > NMF_SUBSAMPLE:
    rng = np.random.default_rng(42)
    idx = rng.choice(activations_np.shape[0], size=NMF_SUBSAMPLE, replace=False)
    activations_fit = activations_np[idx]
else:
    activations_fit = activations_np

nmf = NMF(n_components=N_CLUSTERS, random_state=42, max_iter=500)
W = nmf.fit_transform(activations_fit)  # [N_samples_fit, N_CLUSTERS]
H = nmf.components_                     # [N_CLUSTERS, N_neurons]
neuron_assignments = H.argmax(axis=0) + 1  # 1-indexed cluster IDs
clusters = neuron_assignments

print(f"Total neurons clustered: {activations_np.shape[1]}")
print(f"Clusters assigned: {np.unique(clusters)}")

# Build cluster_map: {cluster_id: [neuron_indices]}
cluster_map = defaultdict(list)
for neuron_idx, cluster_id in enumerate(clusters):
    cluster_map[cluster_id].append(neuron_idx)

## Cluster selectivity analysis

In [ ]:
selectivity_results = funcs.compute_cluster_selectivity(cluster_map, all_neuron_activations, labels)

In [ ]:
funcs.plot_cluster_activation_heatmap(selectivity_results)

## Cluster ablation

In [ ]:
cluster_results = {}
for cluster_id, neuron_indices in cluster_map.items():
    per_class_acc = funcs.cluster_criticality_per_class(
        analysis_model,
        neuron_indices,
        layer_mapping,
        val_loader,
        cluster_id,
        device=device
    )
    cluster_results[cluster_id] = per_class_acc

In [ ]:
print(f"Accuracy: {analysis_model.accuracy(val_loader):.2f}")

In [ ]:
cluster_class_changes = plots.plot_cluster_accuracy_bars(cluster_results, target_labels=list(range(10)))

## Prototype and difference map plots

In [ ]:
# Compute prototypes and difference maps for all clusters
all_prototypes = funcs.compute_prototypes_all_clusters(
    cluster_map=cluster_map,
    all_activations=all_neuron_activations,
    images=images,
    top_frac=0.1,           # fraction of top-activating samples per cluster
    use_global_mean=True     # normalize diff map relative to global mean
)

In [ ]:
plots.plot_cluster_prototypes_and_diff_all(all_prototypes)

## cluster extraction